In [12]:
import os
from sshtunnel import SSHTunnelForwarder
from kafka import KafkaProducer, KafkaConsumer
from dotenv import load_dotenv

# Load environment variables
load_dotenv("./.env")

def create_ssh_tunnel(SSH_HOST, SSH_PORT, SSH_USERNAME, SSH_PASSWORD, KAFKA_HOST, LOCAL_BIND_PORT, REMOTE_BIND_PORT):
    """Establishes SSH Tunnel to connect to the remote Kafka broker."""
    server = SSHTunnelForwarder(
        (SSH_HOST, SSH_PORT),
        ssh_username=SSH_USERNAME,
        ssh_password=SSH_PASSWORD,
        remote_bind_address=(KAFKA_HOST, REMOTE_BIND_PORT),
        local_bind_address=('0.0.0.0', LOCAL_BIND_PORT)
    )
    
    server.start()
    return server

def create_kafka_producer(LOCAL_KAFKA_PORT):
    """Creates a Kafka producer to send messages."""
    producer = KafkaProducer(
        bootstrap_servers=f'127.0.0.1:{LOCAL_KAFKA_PORT}',
        value_serializer=lambda v: v.encode('utf-8')  # Serializes messages to UTF-8
    )
    return producer

def create_kafka_consumer(LOCAL_KAFKA_PORT, topic):
    """Creates a Kafka consumer to receive messages."""
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=f'127.0.0.1:{LOCAL_KAFKA_PORT}',
        auto_offset_reset='earliest',
        enable_auto_commit=True,
        group_id='my-group',
        value_deserializer=lambda x: x.decode('utf-8')  # Deserializes messages from UTF-8
    )
    return consumer

# Example usage with environment variables
SSH_HOST = "152.118.31.54"
SSH_PORT = 36022
SSH_USERNAME = "user01"
SSH_PASSWORD = "pass2024"
MONGO_HOST = "127.0.0.1"
MONGO_DB = "sispro-tews"
LOCAL_BIND_PORT = 9999  # Defaulting to 10022 if not provided
REMOTE_BIND_PORT = 9999  # Default MongoDB port
KAFKA_HOST = "152.118.31.54"
topic = "testing_ssh"
# Create SSH tunnel
ssh_server = create_ssh_tunnel(SSH_HOST, SSH_PORT, SSH_USERNAME, SSH_PASSWORD, KAFKA_HOST, LOCAL_BIND_PORT, REMOTE_BIND_PORT)


# Kafka consumer example
consumer = create_kafka_consumer(LOCAL_BIND_PORT, topic)

# Receive messages from Kafka topic
for message in consumer:
    print(f"Received message: {message.value}")
    break  # Exit after reading one message, remove to keep listening

# Stop SSH server after use
ssh_server.stop()


2024-09-16 12:50:27,109| ERROR   | Problem setting SSH Forwarder up: Couldn't open tunnel 0.0.0.0:9999 <> 152.118.31.54:9999 might be in use or destination not reachable


HandlerSSHTunnelForwarderError: An error occurred while opening tunnels.

In [4]:
from kafka import KafkaProducer
import json

# Initialize Kafka producer with custom broker address
producer = KafkaProducer(
    bootstrap_servers=['127.0.0.1:8321'],  # Replace with your Kafka broker's address
    value_serializer=lambda v: json.dumps(v).encode('utf-8')  # Serialize data to JSON
)

# Sending data to Kafka topic
topic_name = 'my-topic'
message = {'key': 'value', 'another_key': 'another_value'}

producer.send(topic_name, value=message)
producer.flush()  # Ensure all messages are sent before closing the producer

print(f"Message sent to topic '{topic_name}' on broker '152.118.31.54:9999'")

# Close the producer
producer.close()


ValueError: Invalid file object: None